# 01 Single Sample Inference v0.1

**Notebook version:** v0.1  
**Category:** inference  
**Purpose:** Select one trained model run and one raw synthetic sample, then execute a single explicit inference pass through the existing preprocessing path.  
**Inputs:** `./models`, `./input`, `./src/inference_v0_1`  
**Expected outputs:** ROI/model-input image, predicted distance/orientation, actual distance/orientation, deltas, and optional JSON save artifact.  
**Artifact write mode:** optional result JSON under `./output/results`

In [1]:
# Repo Setup
from pathlib import Path
import sys


INFERENCE_ROOT_CANDIDATES = ('05_inference-v0.1', '04_inference-v0.1')


def find_repo_root(start: Path | None = None) -> tuple[Path, Path]:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        for name in INFERENCE_ROOT_CANDIDATES:
            inference_root = current / name
            if (inference_root / 'src').exists() and (inference_root / 'models').exists():
                return current, inference_root

    raise RuntimeError(
        f'Could not locate repo root containing any of: {INFERENCE_ROOT_CANDIDATES}'
    )


repo_root, inference_root = find_repo_root()
src_root = inference_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print(f'repo_root={repo_root}')
print(f'inference_root={inference_root}')


repo_root=/home/mitch/development/raccoon-ball
inference_root=/home/mitch/development/raccoon-ball/05_inference-v0.1


In [2]:
# Control Pane
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

from inference_v0_1 import (
    discover_model_runs,
    discover_raw_corpora,
    list_corpus_image_names,
    run_single_sample_inference,
)

models = discover_model_runs(inference_root / 'models')
if not models:
    raise FileNotFoundError(f'No model run artifacts found under {inference_root / "models"}')

corpora = discover_raw_corpora(inference_root / 'input')
if not corpora:
    raise FileNotFoundError(
        f'No valid raw-image corpora found under {inference_root / "input"}. '
        'Expected corpus folders with images/ plus manifests/run.json and manifests/samples.csv.'
    )

model_by_label = {artifact.label: artifact for artifact in models}
corpus_by_name = {corpus.name: corpus for corpus in corpora}
all_input_dirs = sorted(path.name for path in (inference_root / 'input').iterdir() if path.is_dir())
ignored_dirs = sorted(set(all_input_dirs).difference(corpus_by_name))

print(f'discovered_models={len(models)}')
print(f'discovered_raw_corpora={len(corpora)}')
if ignored_dirs:
    print('ignored_non_raw_input_dirs=', ignored_dirs)

model_select = widgets.Select(
    options=[artifact.label for artifact in models],
    value=models[0].label,
    description='Model',
    rows=min(8, max(4, len(models))),
)
corpus_select = widgets.Select(
    options=[corpus.name for corpus in corpora],
    value=corpora[0].name,
    description='Corpus',
    rows=min(8, max(4, len(corpora))),
)
image_select = widgets.Select(
    options=[],
    value=None,
    description='Image',
    rows=12,
    layout=widgets.Layout(height='280px'),
)
image_name_text = widgets.Text(
    value='',
    description='Image name',
    placeholder='Select an image',
)
run_button = widgets.Button(
    description='Run Inference',
    button_style='primary',
    disabled=True,
)
save_toggle = widgets.Checkbox(value=False, description='Save JSON')
status_html = widgets.HTML()
results_out = widgets.Output()


def clear_results() -> None:
    status_html.value = ''
    with results_out:
        clear_output()


def refresh_image_options(*_args) -> None:
    clear_results()
    corpus_name = corpus_select.value
    if not corpus_name:
        image_select.options = []
        image_select.index = None
        image_name_text.value = ''
        run_button.disabled = True
        return

    image_names = list_corpus_image_names(corpus_by_name[corpus_name])
    image_select.options = image_names
    image_select.rows = 12
    image_select.index = None
    image_name_text.value = ''
    run_button.disabled = True


def on_model_change(change) -> None:
    if change['name'] == 'value':
        clear_results()


def on_corpus_change(change) -> None:
    if change['name'] == 'value':
        refresh_image_options()


def on_image_change(change) -> None:
    if change['name'] != 'value':
        return
    clear_results()
    selected = change['new'] or ''
    image_name_text.value = selected


def on_image_text_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_results()
    run_button.disabled = not bool(str(change['new']).strip())


def on_run_clicked(_button) -> None:
    clear_results()
    selected_model = model_by_label[model_select.value]
    selected_corpus = corpus_by_name[corpus_select.value]
    selected_image = image_name_text.value.strip()
    if not selected_image:
        status_html.value = '<b>Select an image first.</b>'
        return

    run_button.disabled = True
    status_html.value = '<i>Running single-sample inference...</i>'
    try:
        result = run_single_sample_inference(
            selected_model.run_dir,
            selected_corpus.root,
            selected_image,
            save_result=bool(save_toggle.value),
        )
    except Exception as exc:
        status_html.value = (
            f'<span style="color:#b00020;"><b>Inference failed:</b> {exc}</span>'
        )
    else:
        status_html.value = '<b>Inference completed.</b>'
        with results_out:
            clear_output()

            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(result.roi_image, cmap='gray', vmin=0.0, vmax=1.0)
            ax.set_title('ROI / Model Input')
            ax.axis('off')
            plt.show()

            summary_rows = [
                {'field': 'Model', 'value': result.selected_model_label},
                {'field': 'Corpus', 'value': result.selected_corpus_name},
                {'field': 'Image', 'value': result.selected_image_name},
                {'field': 'Predicted distance (m)', 'value': f'{result.predicted_distance_m:.4f}'},
                {'field': 'Actual distance (m)', 'value': f'{result.actual_distance_m:.4f}'},
                {'field': 'Distance delta (m)', 'value': f'{result.distance_delta_m:+.4f}'},
                {'field': 'Absolute distance error (m)', 'value': f'{result.absolute_distance_error_m:.4f}'},
                {'field': 'Predicted orientation (deg)', 'value': f'{result.predicted_orientation_deg:.4f}'},
                {'field': 'Actual orientation (deg)', 'value': f'{result.actual_orientation_deg:.4f}'},
                {'field': 'Orientation delta (deg)', 'value': f'{result.orientation_delta_deg:+.4f}'},
                {'field': 'Absolute orientation error (deg)', 'value': f'{result.absolute_orientation_error_deg:.4f}'},
            ]
            display(pd.DataFrame(summary_rows))

            if result.saved_json_path is not None:
                print(f'Saved JSON: {result.saved_json_path}')
            if result.saved_roi_path is not None:
                print(f'Saved ROI image: {result.saved_roi_path}')
    finally:
        run_button.disabled = not bool(image_name_text.value.strip())


model_select.observe(on_model_change, names='value')
corpus_select.observe(on_corpus_change, names='value')
image_select.observe(on_image_change, names='value')
image_name_text.observe(on_image_text_change, names='value')
run_button.on_click(on_run_clicked)

refresh_image_options()

controls = widgets.VBox(
    [
        widgets.HBox([model_select, corpus_select, image_select]),
        widgets.HBox([image_name_text, run_button, save_toggle]),
        status_html,
        results_out,
    ]
)
display(controls)


FileNotFoundError: No valid raw-image corpora found under /home/mitch/development/raccoon-ball/05_inference-v0.1/input. Expected corpus folders with images/ plus manifests/run.json and manifests/samples.csv.